In [7]:
# ============================================================
# IDENTIFY NEXT BATCH — LOCAL EXECUTION
# ============================================================

import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# Load environment
def run_nb(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    src, _ = PythonExporter().from_notebook_node(nb)
    exec(src, globals())

run_nb("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")

control_file = f"{CONTROL_PATH}/batch_control/data.parquet"

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [8]:
# -----------------------------------------
# 1. Landing batches (folders)
# -----------------------------------------
landing_batches = sorted([
    name for name in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, name))
])

In [9]:
# -----------------------------------------
# 2. Tracked batches
# -----------------------------------------
if os.path.exists(control_file):
    tracked_batches = [
        row.batch_id
        for row in spark.read.parquet(control_file)
            .filter(F.col("status").isin("in_progress", "completed"))
            .select("batch_id").distinct().collect()
    ]
else:
    tracked_batches = []

In [10]:
# -----------------------------------------
# 3. Identify earliest unprocessed batch
# -----------------------------------------
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print("Landing batches :", landing_batches)
print("Tracked batches :", tracked_batches)
print("Next batch      :", next_batch)

Landing batches : ['2025-01']
Tracked batches : []
Next batch      : 2025-01


In [11]:
# -----------------------------------------
# 4. Export values for next notebook
# -----------------------------------------
has_batch = next_batch is not None